In [1]:
import sys
import time
import pathlib
cur_path = pathlib.Path().resolve().parent.absolute()
src_loc = cur_path.joinpath("src")
util_loc = src_loc.joinpath("utils")
sys.path.append(str(cur_path))
sys.path.append(str(src_loc))
sys.path.append(str(util_loc))

In [2]:
# load the libraries and initial settings
import os
import pathlib
import time
import datetime
import subprocess
import pyarrow as pa
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib as mpl
mpl.rc('axes', labelsize=14)
mpl.rc('xtick', labelsize=12)
mpl.rc('ytick', labelsize=12)
pd.set_option('display.float_format', lambda x: '%.3f' % x)
from src.conf import Conf
from src.utils.utils import elapsed_time, save_data, load_data
from src.utils.sql import sql
# Load configuration
confparam_path = cur_path.joinpath("src", "conf", "conf_dev.yml")
dataparam_path = cur_path.joinpath("src", "data", "ctp_dbt_model", "dbt_project.yml")
conf = Conf(confparam_path, dataparam_path )
project_start_time = time.time()

In [9]:
query = """
SELECT 
    claim.claim_key, 
    claim.claim_number,
    party.claim_party_key,
    party_role.party_role_name,
    address_line_1 
        || CASE WHEN address_line_2 IS NOT NULL AND address_line_2 <> '' THEN ', ' || address_line_2 ELSE '' END 
        || CASE WHEN address_line_3 IS NOT NULL AND address_line_3 <> '' THEN ', ' || address_line_3 ELSE '' END 
        || ', ' || city
        || ', ' || state_name
        || ', ' || post_code 
        || ', ' || country_code AS full_address,
    addr.address_line_1,
    addr.address_line_2,
    addr.address_line_3,
    addr.city,
    addr.state_name,
    addr.post_code,
    addr.country_code
FROM dbt_integration_analytics_dia.int_claim_base_features_staging claim
LEFT JOIN dbt_conformed_analytics_dia.conf_claim_party_role_base_features_svoc party_role
    ON claim.claim_key = party_role.claim_key
    AND (party_role.party_role_active_flag IS NULL OR party_role.party_role_active_flag = 'Yes')
    AND party_role.dbt_valid_to IS NULL
LEFT JOIN dbt_conformed_analytics_dia.conf_claim_party_base_features_svoc party
    ON party_role.claim_party_key = party.claim_party_key
    AND (party.active_flag IS NULL OR party.active_flag = 'Yes')
    AND party.dbt_valid_to IS NULL
LEFT JOIN dbt_conformed_analytics_dia.conf_claim_address_role_base_features_svoc addr_role
    ON party.primary_address_key = addr_role.claim_address_role_key
    AND addr_role.address_role = 'primary_address'
    AND addr_role.source_table_entity = 'claim_party'
    AND addr_role.dbt_valid_to IS NULL
LEFT JOIN dbt_conformed_analytics_dia.conf_claim_address_base_features_svoc addr
    ON addr_role.claim_address_role_key = addr.claim_address_key
    AND addr.dbt_valid_to IS NULL
WHERE party_role.party_role_name IN (
        'Lawyer/Solicitor','Legal','Legal Representative','Medical',
        'Medical Centre','Medical Professional','Doctor','Primary Doctor',
        'Psychologist','Bike Repairer','Motor Body Repairer','Claimant'
    )
    AND claim.claim_status_name <> 'Draft'
    AND claim.claim_loss_state IN ('NSW')
    -- AND claim.policy_issue_state = 'NSW'
    AND claim.line_of_business_name = 'Compulsory Third Party'
    AND claim.notify_only_claim_flag = 'No'
    AND claim.claim_lodgement_date BETWEEN '2020-06-01' AND '2024-10-01'
    AND (claim.closed_outcome_name IS NULL OR claim.closed_outcome_name = 'Completed')
    AND claim.dbt_valid_to IS NULL
"""


In [3]:
query = """
WITH ctp_claimant_addr AS (
    SELECT 
        a.claim_number,
        b.claim_exposure_id,
        b.claimant_contact_id,
        b.claimant_contact_name,
        TRIM(
            COALESCE(claimant_primary_address_street_number, '') || ' ' ||
            COALESCE(claimant_primary_address_street_name, '') || ' ' ||
            COALESCE(claimant_primary_address_street_type, '') || ', ' ||
            COALESCE(claimant_primary_address_suburb, '') || ', ' ||
            COALESCE(claimant_primary_address_state_name, '') || ', ' ||
            COALESCE(claimant_primary_address_postcode, '') || ', ' ||
            COALESCE(claimant_primary_address_country, '')
        ) AS claimant_address
    FROM ctx.mv_cc_ci_claim_header_ext a
    JOIN pub_core.mv_claim_exposure_header b
        ON a.claim_number = b.claim_number 
    WHERE 
        a.claim_status_name = 'Open' 
        AND a.ctp_statutory_insurer_state_name IN ('NSW')
        AND a.line_of_business_name = 'Compulsory Third Party'
        AND a.notify_only_claim_flag = 'No'
        AND (a.claim_closed_outcome_name IS NULL OR a.claim_closed_outcome_name = 'Completed')
    GROUP BY 
        a.claim_number,
        b.claim_exposure_id,
        b.claimant_contact_id,
        b.claimant_contact_name,
        TRIM(
            COALESCE(claimant_primary_address_street_number, '') || ' ' ||
            COALESCE(claimant_primary_address_street_name, '') || ' ' ||
            COALESCE(claimant_primary_address_street_type, '') || ', ' ||
            COALESCE(claimant_primary_address_suburb, '') || ', ' ||
            COALESCE(claimant_primary_address_state_name, '') || ', ' ||
            COALESCE(claimant_primary_address_postcode, '') || ', ' ||
            COALESCE(claimant_primary_address_country, '')
        )
),
   
ctp_supplier_addr AS ( 
    SELECT 
        a.claim_number, 
        b.claim_exposure_id,
        name AS supplier_name,
        contact_id,
        role_name AS supplier_role_name,
        TRIM(
            CASE 
                WHEN address_line_1 IS NULL AND address_line_2 IS NULL AND address_line_3 IS NULL AND 
                     address_suburb_name IS NULL AND address_state_name IS NULL THEN ''
                ELSE
                    COALESCE(address_line_1 || ' ', '') ||
                    COALESCE(address_line_2 || ' ', '') ||
                    COALESCE(address_line_3 || ', ', ', ') ||
                    COALESCE(address_suburb_name || ', ', '') ||
                    COALESCE(address_state_name || ', ', '') ||
                    COALESCE(address_post_code || ', ', '') ||
                    COALESCE(address_country_code, '')
            END
        ) AS full_address
    FROM ctx.mv_cc_ci_claim_contact_ext a
    LEFT JOIN pub.mv_ctp_payment b
        ON a.claim_number = b.claim_number 
        AND UPPER(a.name) = UPPER(b.party_name)  
    WHERE 
        role_name IN (
            'Medical', 'Legal', 'Interpreter', 'Doctor', 
            'Remedial Massage Therapist', 'Rehabilitation Provider', 
            'Service Provider', 'Lawyer', 'Occupational Therapist', 
            'Hospital', 'Psychologist', 'Physiotherapist', 
            'Treatment Provider'
        )
        AND b.claim_exposure_id IS NOT NULL
)

SELECT 
    a.claim_number,
    a.claim_exposure_id,
    a.claimant_contact_name AS claimant_name,
    a.claimant_address,
    b.supplier_name,
    b.supplier_role_name,
    b.full_address AS supplier_address
FROM ctp_claimant_addr a
JOIN ctp_supplier_addr b
    ON a.claim_number = b.claim_number
    AND a.claim_exposure_id = b.claim_exposure_id
GROUP BY 
    a.claim_number,
    a.claim_exposure_id,
    a.claimant_contact_name,
    a.claimant_address,
    b.supplier_name,
    b.supplier_role_name,
    b.full_address;
"""


In [4]:
print('*' * 60)
print("Data query started ...")
function_start_time = time.time()
df = sql( conf=conf, fn="get", sql=query)
print('*'*60)
print("data query finished")
elapsed_time('Query data from EDH', project_start_time, function_start_time)

************************************************************
Data query started ...
************************************************************
data query finished

Elapsed time (Query data from EDH): 0:1:54.926
Total elapsed time: 0:2:11.861


In [5]:
import pandas as pd 
import os, requests, sys 

df

,claim_number,claim_exposure_id,claimant_name,claimant_address,supplier_name,supplier_role_name,supplier_address
0,NWRND1800109,6,JAIME VALENCIA,"49 Avalon Court, PHILLIP, ACT, 2606, Australia",acacia healthcare pty ltd,Service Provider,"PO Box 3329 , ST PAULS, NSW, 2031, AU"
1,NWRND1800164,1,Joshua Cox,"37 Menangle STREET, GANMAIN, NSW, 2702, Australia",gerard malouf and partners solicitors,Legal,"PO BOX 463 , PARRAMATTA, NSW, 2124, AU"
2,NWRND1900051,2,NASSOUH EL DEBAL,"31 SACKVILLE ST, FAIRFIELD, NSW, 2165, Australia",fit by physio pty ltd,Service Provider,"Shop 2, 363 Beamish Street , CAMPSIE, NSW, 219..."
3,NWRND1900055,1,Mildred Villapando,"75 Kleins Rd, NORTHMEAD, NSW, 2152, Australia",exphys rehab pty ltd,Service Provider,"PO Box 2154 , Woolooware, NSW, 2230, AU"
4,NWRND1900118,1,Nicole Shorrock,"2 Inkerman Rd, EMU HEIGHTS, NSW, 2750, Australia",rosalind dayman,Service Provider,"Genesis, Ste 35 76 Rawson St , EPPING, NSW, 21..."
...,...,...,...,...,...,...,...
79306,NWRTP2403482,1,Dianne Fernando,"17 Prospero St, MARYLAND, NSW, 2287, Australia",practitioners trust account no 26,Service Provider,"Dr Syed Hasan PO Box 131 , LEICHHARDT, NSW, 20..."
79307,NWRTP2403495,1,Adam Muirhead,"10 Lonhro Wy, PORT MACQUARIE, NSW, 2444, Austr...",mid north coast local health district,Service Provider,"PO Box 126 , PORT MACQUARIE, NSW, 2444, AU"
79308,NWRTP2500056,1,Cheston Wu,"172 Gardeners Rd, KINGSFORD, NSW, 2032, Australia",south eastern sydney local health district,Service Provider,"Finance and Corporate Services Locked Bag 21 ,..."
79309,NWRTP2500072,1,Joon Hyuk Choi,"11 Settlers Blvd, LIBERTY GROVE, NSW, 2138, Au...",j medical & cosmetic centre,Service Provider,1-112 Lidcombe Shopping Centre 92 Parramatta R...


In [9]:
class GeoCodeAPI(object): 

    def __init__ (self, df, name_of_address_column = 'fulladdress', list_of_cols_to_attach = ['longitude','latitude','reliability','ext_GEO_ID'], proxies={}): 
        self.proxies = proxies
        self.list_of_cols_to_attach = list_of_cols_to_attach         
        self.GC = self.Sequential(list(df[name_of_address_column]))
        self.df = df.join(pd.DataFrame(self.GC, columns = self.list_of_cols_to_attach))
    def Sequential(self,list_of_addresses): 
        return [self.ProcessOne(i) for i in [(n,v) for n, v in enumerate(list_of_addresses)]] # packing into tuples (N, address)
    def ProcessOne(self,data): 
        self.url = r'http://ulpcrd001:8188/geosingle'
        # self.url = r'http://ulpcrd001:8188/single'
        
        self.headers = {r'Content-Type': 'application/json', 'output_mode' : 'json', 'Accept' : 'application/json'} 
        idN, address = data         
        rj = requests.post(url = self.url, data = address, headers = self.headers, json = True, proxies = self.proxies).json()
        return [rj[i] for i in self.list_of_cols_to_attach ]

In [19]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 86218 entries, 0 to 86217
Data columns (total 12 columns):
 #   Column           Non-Null Count  Dtype 
---  ------           --------------  ----- 
 0   claim_key        86218 non-null  object
 1   claim_number     86218 non-null  object
 2   claim_party_key  83718 non-null  object
 3   party_role_name  86218 non-null  object
 4   full_address     61443 non-null  object
 5   address_line_1   61659 non-null  object
 6   address_line_2   6846 non-null   object
 7   address_line_3   0 non-null      object
 8   city             61689 non-null  object
 9   state_name       61628 non-null  object
 10  post_code        61605 non-null  object
 11  country_code     64197 non-null  object
dtypes: object(12)
memory usage: 7.9+ MB


In [36]:
df = df.dropna(subset=['full_address'])

In [8]:
df.to_csv('address.csv')

In [13]:
print('*' * 60)
print("Geocoding started ...")
function_start_time = time.time()
gc = GeoCodeAPI (df, name_of_address_column = 'claimant_address', list_of_cols_to_attach = ['longitude','latitude','ext_GEO_ID', 'gnafPID'] )
print('*'*60)
print("Geocoding ended")
elapsed_time('Geocoding time', project_start_time, function_start_time)


************************************************************
Geocoding started ...
************************************************************
Geocoding ended

Elapsed time (Geocoding time): 1:33:53.554
Total elapsed time: 2:41:04.649


In [14]:
gc.df

,claim_number,claim_exposure_id,claimant_name,claimant_address,supplier_name,supplier_role_name,supplier_address,longitude,latitude,ext_GEO_ID,gnafPID
0,NWRND1800109,6,JAIME VALENCIA,"49 Avalon Court, PHILLIP, ACT, 2606, Australia",acacia healthcare pty ltd,Service Provider,"PO Box 3329 , ST PAULS, NSW, 2031, AU",149.09821159,-35.34900670,AGAACT717737809,GAACT717737809
1,NWRND1800164,1,Joshua Cox,"37 Menangle STREET, GANMAIN, NSW, 2702, Australia",gerard malouf and partners solicitors,Legal,"PO BOX 463 , PARRAMATTA, NSW, 2124, AU",147.03700256,-34.79684448,AGANSW704808475,GANSW704808475
2,NWRND1900051,2,NASSOUH EL DEBAL,"31 SACKVILLE ST, FAIRFIELD, NSW, 2165, Australia",fit by physio pty ltd,Service Provider,"Shop 2, 363 Beamish Street , CAMPSIE, NSW, 219...",150.94946806,-33.86591204,AGANSW716741848,GANSW716741848
3,NWRND1900055,1,Mildred Villapando,"75 Kleins Rd, NORTHMEAD, NSW, 2152, Australia",exphys rehab pty ltd,Service Provider,"PO Box 2154 , Woolooware, NSW, 2230, AU",150.99294760,-33.79031497,AGANSW705571192,GANSW705571192
4,NWRND1900118,1,Nicole Shorrock,"2 Inkerman Rd, EMU HEIGHTS, NSW, 2750, Australia",rosalind dayman,Service Provider,"Genesis, Ste 35 76 Rawson St , EPPING, NSW, 21...",150.64671000,-33.73388000,AGANSW704686594,GANSW704686594
...,...,...,...,...,...,...,...,...,...,...,...
79306,NWRTP2403482,1,Dianne Fernando,"17 Prospero St, MARYLAND, NSW, 2287, Australia",practitioners trust account no 26,Service Provider,"Dr Syed Hasan PO Box 131 , LEICHHARDT, NSW, 20...",151.65596647,-32.88126450,AGANSW713187374,GANSW713187374
79307,NWRTP2403495,1,Adam Muirhead,"10 Lonhro Wy, PORT MACQUARIE, NSW, 2444, Austr...",mid north coast local health district,Service Provider,"PO Box 126 , PORT MACQUARIE, NSW, 2444, AU",152.84962434,-31.46509546,AGANSW720016596,GANSW720016596
79308,NWRTP2500056,1,Cheston Wu,"172 Gardeners Rd, KINGSFORD, NSW, 2032, Australia",south eastern sydney local health district,Service Provider,"Finance and Corporate Services Locked Bag 21 ,...",151.21893046,-33.92476959,AGANSW705104188,GANSW705104188
79309,NWRTP2500072,1,Joon Hyuk Choi,"11 Settlers Blvd, LIBERTY GROVE, NSW, 2138, Au...",j medical & cosmetic centre,Service Provider,1-112 Lidcombe Shopping Centre 92 Parramatta R...,151.08490091,-33.84214824,AGANSW711004220,GANSW711004220


In [15]:
print('*' * 60)
print("Geocoding started ...")
function_start_time = time.time()
gc2 = GeoCodeAPI (df, name_of_address_column = 'supplier_address', list_of_cols_to_attach = ['longitude','latitude','ext_GEO_ID', 'gnafPID'] )
print('*'*60)
print("Geocoding ended")
elapsed_time('Geocoding time', project_start_time, function_start_time)

************************************************************
Geocoding started ...
************************************************************
Geocoding ended

Elapsed time (Geocoding time): 1:28:19.914
Total elapsed time: 49:27:52.878


In [18]:
gc2.df

,claim_number,claim_exposure_id,claimant_name,claimant_address,supplier_name,supplier_role_name,supplier_address,longitude,latitude,ext_GEO_ID,gnafPID
0,NWRND1800109,6,JAIME VALENCIA,"49 Avalon Court, PHILLIP, ACT, 2606, Australia",acacia healthcare pty ltd,Service Provider,"PO Box 3329 , ST PAULS, NSW, 2031, AU",,,,
1,NWRND1800164,1,Joshua Cox,"37 Menangle STREET, GANMAIN, NSW, 2702, Australia",gerard malouf and partners solicitors,Legal,"PO BOX 463 , PARRAMATTA, NSW, 2124, AU",151.00687829,-33.81396457,Lloc581d5d75cd4f,loc581d5d75cd4f
2,NWRND1900051,2,NASSOUH EL DEBAL,"31 SACKVILLE ST, FAIRFIELD, NSW, 2165, Australia",fit by physio pty ltd,Service Provider,"Shop 2, 363 Beamish Street , CAMPSIE, NSW, 219...",151.10442986,-33.91537277,AGANSW718684557,GANSW718684557
3,NWRND1900055,1,Mildred Villapando,"75 Kleins Rd, NORTHMEAD, NSW, 2152, Australia",exphys rehab pty ltd,Service Provider,"PO Box 2154 , Woolooware, NSW, 2230, AU",151.14052318,-34.04282203,Lloc230e97dd535c,loc230e97dd535c
4,NWRND1900118,1,Nicole Shorrock,"2 Inkerman Rd, EMU HEIGHTS, NSW, 2750, Australia",rosalind dayman,Service Provider,"Genesis, Ste 35 76 Rawson St , EPPING, NSW, 21...",151.08085928,-33.77183715,AGANSW717223283,GANSW717223283
...,...,...,...,...,...,...,...,...,...,...,...
79306,NWRTP2403482,1,Dianne Fernando,"17 Prospero St, MARYLAND, NSW, 2287, Australia",practitioners trust account no 26,Service Provider,"Dr Syed Hasan PO Box 131 , LEICHHARDT, NSW, 20...",151.15476218,-33.88241727,Lloc6dc88e405737,loc6dc88e405737
79307,NWRTP2403495,1,Adam Muirhead,"10 Lonhro Wy, PORT MACQUARIE, NSW, 2444, Austr...",mid north coast local health district,Service Provider,"PO Box 126 , PORT MACQUARIE, NSW, 2444, AU",152.89308622,-31.45490157,Lloc57ade01a1760,loc57ade01a1760
79308,NWRTP2500056,1,Cheston Wu,"172 Gardeners Rd, KINGSFORD, NSW, 2032, Australia",south eastern sydney local health district,Service Provider,"Finance and Corporate Services Locked Bag 21 ,...",151.12549800,-34.01874600,Lloc564fee4d62f2,loc564fee4d62f2
79309,NWRTP2500072,1,Joon Hyuk Choi,"11 Settlers Blvd, LIBERTY GROVE, NSW, 2138, Au...",j medical & cosmetic centre,Service Provider,1-112 Lidcombe Shopping Centre 92 Parramatta R...,,,,


In [22]:
df1=gc.df.rename(
    columns={'longitude': 'claimant_longitude', 'latitude': 'claimant_latitude'}
)
df2 = gc2.df[['longitude', 'latitude']].rename(
    columns={'longitude': 'supplier_longitude', 'latitude': 'supplier_latitude'}
)

In [23]:
df_geo=pd.concat([df1, df2], axis=1)
df_geo

,claim_number,claim_exposure_id,claimant_name,claimant_address,supplier_name,supplier_role_name,supplier_address,claimant_longitude,claimant_latitude,ext_GEO_ID,gnafPID,supplier_longitude,supplier_latitude
0,NWRND1800109,6,JAIME VALENCIA,"49 Avalon Court, PHILLIP, ACT, 2606, Australia",acacia healthcare pty ltd,Service Provider,"PO Box 3329 , ST PAULS, NSW, 2031, AU",149.09821159,-35.34900670,AGAACT717737809,GAACT717737809,,
1,NWRND1800164,1,Joshua Cox,"37 Menangle STREET, GANMAIN, NSW, 2702, Australia",gerard malouf and partners solicitors,Legal,"PO BOX 463 , PARRAMATTA, NSW, 2124, AU",147.03700256,-34.79684448,AGANSW704808475,GANSW704808475,151.00687829,-33.81396457
2,NWRND1900051,2,NASSOUH EL DEBAL,"31 SACKVILLE ST, FAIRFIELD, NSW, 2165, Australia",fit by physio pty ltd,Service Provider,"Shop 2, 363 Beamish Street , CAMPSIE, NSW, 219...",150.94946806,-33.86591204,AGANSW716741848,GANSW716741848,151.10442986,-33.91537277
3,NWRND1900055,1,Mildred Villapando,"75 Kleins Rd, NORTHMEAD, NSW, 2152, Australia",exphys rehab pty ltd,Service Provider,"PO Box 2154 , Woolooware, NSW, 2230, AU",150.99294760,-33.79031497,AGANSW705571192,GANSW705571192,151.14052318,-34.04282203
4,NWRND1900118,1,Nicole Shorrock,"2 Inkerman Rd, EMU HEIGHTS, NSW, 2750, Australia",rosalind dayman,Service Provider,"Genesis, Ste 35 76 Rawson St , EPPING, NSW, 21...",150.64671000,-33.73388000,AGANSW704686594,GANSW704686594,151.08085928,-33.77183715
...,...,...,...,...,...,...,...,...,...,...,...,...,...
79306,NWRTP2403482,1,Dianne Fernando,"17 Prospero St, MARYLAND, NSW, 2287, Australia",practitioners trust account no 26,Service Provider,"Dr Syed Hasan PO Box 131 , LEICHHARDT, NSW, 20...",151.65596647,-32.88126450,AGANSW713187374,GANSW713187374,151.15476218,-33.88241727
79307,NWRTP2403495,1,Adam Muirhead,"10 Lonhro Wy, PORT MACQUARIE, NSW, 2444, Austr...",mid north coast local health district,Service Provider,"PO Box 126 , PORT MACQUARIE, NSW, 2444, AU",152.84962434,-31.46509546,AGANSW720016596,GANSW720016596,152.89308622,-31.45490157
79308,NWRTP2500056,1,Cheston Wu,"172 Gardeners Rd, KINGSFORD, NSW, 2032, Australia",south eastern sydney local health district,Service Provider,"Finance and Corporate Services Locked Bag 21 ,...",151.21893046,-33.92476959,AGANSW705104188,GANSW705104188,151.12549800,-34.01874600
79309,NWRTP2500072,1,Joon Hyuk Choi,"11 Settlers Blvd, LIBERTY GROVE, NSW, 2138, Au...",j medical & cosmetic centre,Service Provider,1-112 Lidcombe Shopping Centre 92 Parramatta R...,151.08490091,-33.84214824,AGANSW711004220,GANSW711004220,,


In [36]:
import numpy as np
df_geo=df_geo.replace('', np.nan)
df_geo.isna().any()

claim_number          False
claim_exposure_id     False
claimant_name         False
claimant_address      False
supplier_name         False
supplier_role_name    False
supplier_address      False
claimant_longitude     True
claimant_latitude      True
ext_GEO_ID             True
gnafPID                True
supplier_longitude     True
supplier_latitude      True
dtype: bool

In [37]:
df_geo = df_geo.dropna(
    subset=['claimant_longitude', 'claimant_latitude', 'supplier_longitude', 'supplier_latitude']
)
df_geo

,claim_number,claim_exposure_id,claimant_name,claimant_address,supplier_name,supplier_role_name,supplier_address,claimant_longitude,claimant_latitude,ext_GEO_ID,gnafPID,supplier_longitude,supplier_latitude
1,NWRND1800164,1,Joshua Cox,"37 Menangle STREET, GANMAIN, NSW, 2702, Australia",gerard malouf and partners solicitors,Legal,"PO BOX 463 , PARRAMATTA, NSW, 2124, AU",147.03700256,-34.79684448,AGANSW704808475,GANSW704808475,151.00687829,-33.81396457
2,NWRND1900051,2,NASSOUH EL DEBAL,"31 SACKVILLE ST, FAIRFIELD, NSW, 2165, Australia",fit by physio pty ltd,Service Provider,"Shop 2, 363 Beamish Street , CAMPSIE, NSW, 219...",150.94946806,-33.86591204,AGANSW716741848,GANSW716741848,151.10442986,-33.91537277
3,NWRND1900055,1,Mildred Villapando,"75 Kleins Rd, NORTHMEAD, NSW, 2152, Australia",exphys rehab pty ltd,Service Provider,"PO Box 2154 , Woolooware, NSW, 2230, AU",150.99294760,-33.79031497,AGANSW705571192,GANSW705571192,151.14052318,-34.04282203
4,NWRND1900118,1,Nicole Shorrock,"2 Inkerman Rd, EMU HEIGHTS, NSW, 2750, Australia",rosalind dayman,Service Provider,"Genesis, Ste 35 76 Rawson St , EPPING, NSW, 21...",150.64671000,-33.73388000,AGANSW704686594,GANSW704686594,151.08085928,-33.77183715
5,NWRND1900171,1,Darren Hunt,"21 Hereford St, HOBARTVILLE, NSW, 2753, Australia",penrith physiotherapy sports centre,Service Provider,Nepean Private Specialist Centre 1A Barber Ave...,150.74543349,-33.60064062,AGANSW704965734,GANSW704965734,150.71376766,-33.75785621
...,...,...,...,...,...,...,...,...,...,...,...,...,...
79305,NWRTP2403433,1,Sean Saap,"7 Gladys St, RYDALMERE, NSW, 2116, Australia",waterfront medical centre,Service Provider,"Shop 4E, 4 The Piazza , WENTWORTH POINT, NSW, ...",151.04622561,-33.81449465,AGANSW707649568,GANSW707649568,151.07424520,-33.83191484
79306,NWRTP2403482,1,Dianne Fernando,"17 Prospero St, MARYLAND, NSW, 2287, Australia",practitioners trust account no 26,Service Provider,"Dr Syed Hasan PO Box 131 , LEICHHARDT, NSW, 20...",151.65596647,-32.88126450,AGANSW713187374,GANSW713187374,151.15476218,-33.88241727
79307,NWRTP2403495,1,Adam Muirhead,"10 Lonhro Wy, PORT MACQUARIE, NSW, 2444, Austr...",mid north coast local health district,Service Provider,"PO Box 126 , PORT MACQUARIE, NSW, 2444, AU",152.84962434,-31.46509546,AGANSW720016596,GANSW720016596,152.89308622,-31.45490157
79308,NWRTP2500056,1,Cheston Wu,"172 Gardeners Rd, KINGSFORD, NSW, 2032, Australia",south eastern sydney local health district,Service Provider,"Finance and Corporate Services Locked Bag 21 ,...",151.21893046,-33.92476959,AGANSW705104188,GANSW705104188,151.12549800,-34.01874600


In [39]:
df_geo.to_csv('geocoded_address.csv', index=False)

In [26]:
list_of_addresses=list(df['full_address'])

In [32]:
for i in [(n,v) for n, v in enumerate(list_of_addresses)]:
        url = r'http://ulpcrd001:8188/geosingle'
        
        headers = {r'Content-Type': 'application/json', 'output_mode' : 'json', 'Accept' : 'application/json'} 
        idN, address=i
        rj = requests.post(url = url, data = address, headers = headers, json = True, proxies = {}).json()
        print(rj)
        

{'input': '', 'realAddress': '"411 PACIFIC HIGHWAY,BELMONT NORTH NSW 2280"', 'timeTotal': 0, 'timeHttp': 8, 'timeParse': 0, 'timeSql': 4, 'success': True, 'addressQuality': '8', 'floorType': '', 'floorNumber': '', 'country': 'AU', 'localitySummary': '', 'state': 'NSW', 'region': '', 'reliability': '2', 'postalID': '', 'gnafLocalityPID': 'loc031ff1af63b8', 'gnafGroupPID': 'NSW2933604', 'postcode': '2280', 'suburb': 'Belmont North', 'buildingName': '', 'premisesType': '', 'subdwellingType': '', 'subdwellingNumber': '', 'street_Name': 'Pacific', 'street_Type': 'HWY', 'streetDirection': '', 'street_Number': '411', 'streetNumberFrom': '', 'streetNumberTo': '', 'streetNumberSeperator': '', 'ruralNumber': '', 'ruralDelivery': '', 'postalType': '', 'postalNumber': '', 'start_DATE': '2021-08-04 00:00:00.0', 'transaction_ID': '1710402306', 'gnafPID': 'GANSW704073891', 'gid': '340846', 'version': '99999999', 'ext_GEO_ID': 'AGANSW704073891', 'data_VERSION': 'GNAF 2021.2', 'data_SOURCE': 'A', 'lati

KeyboardInterrupt: 